# GitHub PR/Issue Operator - примеры запросов

Этот ноутбук запускает LangChain-агента и отправляет команды в GitHub API.
Перед запуском убедись, что в `.env` заданы `ANTHROPIC_API_KEY` и `GITHUB_TOKEN`.

Внимание: команды ниже создают, комментируют и закрывают реальные issues.


In [1]:
# Check Jupyter interpreter and require Python 3.13
import sys
print(sys.executable)
print(sys.version)
if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Select a Python 3.13 kernel in Jupyter (Kernel -> Change Kernel). Current: " + sys.version)


C:\Users\admin\AppData\Local\Programs\Python\Python313\python.exe
3.13.11 (tags/v3.13.11:6278944, Dec  5 2025, 16:26:58) [MSC v.1944 64 bit (AMD64)]


In [2]:
# Установка зависимостей прямо в текущее ядро
import sys
!{sys.executable} -m pip install -r requirements.txt


In [3]:
# Убедимся, что рабочая директория - корень репозитория
import os
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for path in [start] + list(start.parents):
        if (path / 'requirements.txt').exists() and (path / 'prompts' / 'system.md').exists():
            return path
    return start

REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)


Repo root: D:\Dev\OtusAiForDevs\hw04


In [4]:
from dotenv import load_dotenv

load_dotenv()

assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY is missing in .env'
assert os.getenv('GITHUB_TOKEN'), 'GITHUB_TOKEN is missing in .env'

print('ANTHROPIC_API_KEY set:', bool(os.getenv('ANTHROPIC_API_KEY')))
print('GITHUB_TOKEN set:', bool(os.getenv('GITHUB_TOKEN')))


ANTHROPIC_API_KEY set: True
GITHUB_TOKEN set: True


In [5]:
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic

from tools.github_tool import create_issue, list_issues, comment_issue, close_issue

SYSTEM_PROMPT = Path('prompts/system.md').read_text(encoding='utf-8')
REPO = 'TheJubadze/OtusAiForDevs'  # замени на свой репозиторий

def build_agent(model: str | None = None):
    model = model or os.getenv('ANTHROPIC_MODEL', 'claude-opus-4-5')
    llm = ChatAnthropic(model=model, temperature=0)
    return create_agent(
        model=llm,
        tools=[create_issue, list_issues, comment_issue, close_issue],
        system_prompt=SYSTEM_PROMPT,
    )

agent = build_agent()

def run_query(query: str) -> str:
    last_message = None
    
    for step in agent.stream(
        {'messages': [{'role': 'user', 'content': query}]},
        stream_mode="values",
    ):
        messages = step.get('messages', [])
        if messages:
            last_message = messages[-1]
            last_message.pretty_print()  # печатаем каждый шаг
    
    # Возвращаем контент последнего сообщения
    if last_message is None:
        return ''
    return getattr(last_message, 'content', str(last_message))


## Примеры запросов
Запускай по одному (они делают реальные изменения).


In [6]:
run_query(f"Создай issue в репозитории {REPO} с названием 'тест' и текстом 'Проверка агента'")


================================ Human Message =================================

Создай issue в репозитории TheJubadze/OtusAiForDevs с названием 'тест' и текстом 'Проверка агента'
================================== Ai Message ==================================

[{'id': 'toolu_01WENozFNVX33CF8BDMMCA4R', 'input': {'repo': 'TheJubadze/OtusAiForDevs', 'title': 'тест', 'body': 'Проверка агента'}, 'name': 'create_issue', 'type': 'tool_use'}]
Tool Calls:
  create_issue (toolu_01WENozFNVX33CF8BDMMCA4R)
 Call ID: toolu_01WENozFNVX33CF8BDMMCA4R
  Args:
    repo: TheJubadze/OtusAiForDevs
    title: тест
    body: Проверка агента
[github_tool] result={"status": "success", "action": "create_issue", "data": {"number": 60, "title": "\u0442\u0435\u0441\u0442", "state": "open", "url": "https://github.com/TheJubadze/OtusAiForDevs/issues/60", "labels": []}, "http_status": 201}
================================= Tool Message =================================
Name: create_issue

{"status": "success", "acti

'Status: success\nAction: Created issue #60 in TheJubadze/OtusAiForDevs\nData: Issue #60 "тест" создан успешно. URL: https://github.com/TheJubadze/OtusAiForDevs/issues/60\nErrors: -'

In [7]:
run_query(f"Покажи 5 открытых issue с label tech-debt в репозитории {REPO}")


================================ Human Message =================================

Покажи 5 открытых issue с label tech-debt в репозитории TheJubadze/OtusAiForDevs
================================== Ai Message ==================================

[{'id': 'toolu_01Dzf7wLbesEmTTR8hWqX9zj', 'input': {'repo': 'TheJubadze/OtusAiForDevs', 'state': 'open', 'label': 'tech-debt', 'limit': 5}, 'name': 'list_issues', 'type': 'tool_use'}]
Tool Calls:
  list_issues (toolu_01Dzf7wLbesEmTTR8hWqX9zj)
 Call ID: toolu_01Dzf7wLbesEmTTR8hWqX9zj
  Args:
    repo: TheJubadze/OtusAiForDevs
    state: open
    label: tech-debt
    limit: 5
[github_tool] result={"status": "success", "action": "list_issues", "data": [], "http_status": 200}
================================= Tool Message =================================
Name: list_issues

{"status": "success", "action": "list_issues", "data": [], "http_status": 200}
================================== Ai Message ==================================

Status: success
A

"Status: success\nAction: Listed open issues with label 'tech-debt'\nData: Не найдено открытых issues с меткой 'tech-debt' в репозитории TheJubadze/OtusAiForDevs\nErrors: -"

In [8]:
run_query(f"Добавь комментарий к issue 1 в {REPO}: 'Проверка комментария от агента'")


================================ Human Message =================================

Добавь комментарий к issue 1 в TheJubadze/OtusAiForDevs: 'Проверка комментария от агента'
================================== Ai Message ==================================

[{'id': 'toolu_016jU5vZuRM1S9y5hixZiQz1', 'input': {'repo': 'TheJubadze/OtusAiForDevs', 'issue_number': 1, 'comment': 'Проверка комментария от агента'}, 'name': 'comment_issue', 'type': 'tool_use'}]
Tool Calls:
  comment_issue (toolu_016jU5vZuRM1S9y5hixZiQz1)
 Call ID: toolu_016jU5vZuRM1S9y5hixZiQz1
  Args:
    repo: TheJubadze/OtusAiForDevs
    issue_number: 1
    comment: Проверка комментария от агента
[github_tool] result={"status": "success", "action": "comment_issue", "data": {"id": 3810409346, "url": "https://github.com/TheJubadze/OtusAiForDevs/issues/1#issuecomment-3810409346", "body": "\u041f\u0440\u043e\u0432\u0435\u0440\u043a\u0430 \u043a\u043e\u043c\u043c\u0435\u043d\u0442\u0430\u0440\u0438\u044f \u043e\u0442 \u0430\u0433\u04

'Status: success\nAction: Added comment to issue #1 in TheJubadze/OtusAiForDevs\nData: Comment posted successfully (ID: 3810409346) - https://github.com/TheJubadze/OtusAiForDevs/issues/1#issuecomment-3810409346\nErrors: -'

In [9]:
run_query(f"Закрой issue 1 в репозитории {REPO}")


================================ Human Message =================================

Закрой issue 1 в репозитории TheJubadze/OtusAiForDevs
================================== Ai Message ==================================

[{'id': 'toolu_01WeBmBEuB6NtSkHt2NixoAg', 'input': {'repo': 'TheJubadze/OtusAiForDevs', 'issue_number': 1}, 'name': 'close_issue', 'type': 'tool_use'}]
Tool Calls:
  close_issue (toolu_01WeBmBEuB6NtSkHt2NixoAg)
 Call ID: toolu_01WeBmBEuB6NtSkHt2NixoAg
  Args:
    repo: TheJubadze/OtusAiForDevs
    issue_number: 1
[github_tool] result={"status": "success", "action": "close_issue", "data": {"number": 1, "title": "\u0442\u0435\u0441\u0442", "state": "closed", "url": "https://github.com/TheJubadze/OtusAiForDevs/issues/1"}, "http_status": 200}
================================= Tool Message =================================
Name: close_issue

{"status": "success", "action": "close_issue", "data": {"number": 1, "title": "тест", "state": "closed", "url": "https://github.com/TheJu

'Status: success\nAction: Closed issue #1 in TheJubadze/OtusAiForDevs\nData: Issue #1 "тест" закрыт успешно (https://github.com/TheJubadze/OtusAiForDevs/issues/1)\nErrors: -'

In [12]:
run_query(f"Покажи все issues в репозитории {REPO}")


================================ Human Message =================================

Покажи все issues в репозитории TheJubadze/OtusAiForDevs
================================== Ai Message ==================================

[{'id': 'toolu_011K7TXFAzf4f1yYAviCuRqT', 'input': {'repo': 'TheJubadze/OtusAiForDevs', 'limit': 100}, 'name': 'list_issues', 'type': 'tool_use'}]
Tool Calls:
  list_issues (toolu_011K7TXFAzf4f1yYAviCuRqT)
 Call ID: toolu_011K7TXFAzf4f1yYAviCuRqT
  Args:
    repo: TheJubadze/OtusAiForDevs
    limit: 100
[github_tool] result={"status": "success", "action": "list_issues", "data": [{"number": 66, "title": "\u0410\u043d\u043d\u0430 \u041d\u043e\u0432\u0438\u043a\u043e\u0432\u0430", "state": "open", "url": "https://github.com/TheJubadze/OtusAiForDevs/issues/66", "labels": []}, {"number": 59, "title": "\u0422\u0430\u0442\u044c\u044f\u043d\u0430 \u041f\u0430\u0432\u043b\u043e\u0432\u0430", "state": "open", "url": "https://github.com/TheJubadze/OtusAiForDevs/issues/59", "label

'Status: success\nAction: Retrieved all open issues from repository TheJubadze/OtusAiForDevs\n\nData: Найдено 51 открытых issue:\n1. #66 — Анна Новикова\n2. #59 — Татьяна Павлова\n3. #57 — Сергей Новиков\n4. #52 — Мария Иванова\n5. #51 — Наталья Соколова\n6. #49 — тест\n7. #48 — Татьяна Павлова\n8. #46 — Алексей Смирнов\n9. #44 — Иван Лебедев\n10. #43 — Мария Иванова\n11. #42 — Андрей Волков\n12. #41 — Наталья Соколова\n13. #40 — Елена Козлова\n14. #39 — Сергей Новиков\n15. #38 — тест\n16. #37 — тест\n17. #36 — тест\n18. #35 — Татьяна Павлова\n19. #34 — Алексей Смирнов\n20. #33 — Мария Иванова\n21. #32 — Елена Козлова\n22. #31 — Дмитрий Петров\n23. #30 — Сергей Новиков\n24. #29 — Андрей Волков\n25. #28 — Наталья Соколова\n26. #27 — Ольга Морозова\n27. #26 — Иван Лебедев\n28. #25 — тест\n29. #24 — тест\n30. #23 — Татьяна Павлова\n31. #22 — Сергей Новиков\n32. #21 — Иван Лебедев\n33. #20 — Ольга Морозова\n34. #19 — Наталья Соколова\n35. #17 — Андрей Волков\n36. #18 — Дмитрий Петров\n37. 

In [11]:
run_query(f"Создай 10 issue в репозитории {REPO} с названием случайное имя и фамилия и текстом 'Проверка агента + Имя + Фамилия'")

================================ Human Message =================================

Создай 10 issue в репозитории TheJubadze/OtusAiForDevs с названием случайное имя и фамилия и текстом 'Проверка агента + Имя + Фамилия'
================================== Ai Message ==================================

[{'id': 'toolu_01TxGUQ9Rfq1hvz2AEnj8gPU', 'input': {'repo': 'TheJubadze/OtusAiForDevs', 'title': 'Алексей Смирнов', 'body': 'Проверка агента + Алексей + Смирнов'}, 'name': 'create_issue', 'type': 'tool_use'}, {'id': 'toolu_01HD3X2CjM3gNJv2BVRnZjKK', 'input': {'repo': 'TheJubadze/OtusAiForDevs', 'title': 'Мария Иванова', 'body': 'Проверка агента + Мария + Иванова'}, 'name': 'create_issue', 'type': 'tool_use'}, {'id': 'toolu_01HLjtZAJiqdLGnsLokGrt9n', 'input': {'repo': 'TheJubadze/OtusAiForDevs', 'title': 'Дмитрий Петров', 'body': 'Проверка агента + Дмитрий + Петров'}, 'name': 'create_issue', 'type': 'tool_use'}, {'id': 'toolu_016yWidDdzHQiSbBBBi9o5SP', 'input': {'repo': 'TheJubadze/OtusAiForDe

'Status: success\nAction: Создано 10 issues в репозитории TheJubadze/OtusAiForDevs\n\nData: Созданные issues:\n1. #62 — Алексей Смирнов\n2. #65 — Мария Иванова\n3. #69 — Дмитрий Петров\n4. #64 — Екатерина Соколова\n5. #68 — Иван Козлов\n6. #66 — Анна Новикова\n7. #63 — Сергей Морозов\n8. #61 — Ольга Волкова\n9. #67 — Николай Лебедев\n10. #70 — Татьяна Семенова\n\nErrors: -'